In [30]:
# Install once if needed: %pip install torch torchvision matplotlib pandas tqdm
import copy, math, random, time
from dataclasses import dataclass, asdict
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE, '| torch:', torch.__version__)

device: cuda | torch: 2.11.0+cu128


In [31]:
def seed_everything(seed=0):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

@dataclass
class Config:
    dataset: str = 'CIFAR10'
    n_tasks: int = 5
    buffer_size: int = 200
    epochs: int = 50
    batch_size: int = 32
    minibatch_size: int = 32
    lr: float = 0.1
    momentum: float = 0.9
    weight_decay: float = 5e-4
    empty_probability: float = 0.9
    alpha: float = 0.5       # paper Table 9 default for L_ide
    beta: float = 0.5        # paper Table 9 default for L_rep-ice
    class_balance: bool = False
    num_workers: int = 2
    data_root: str = './data'
    gcil_mode: str = 'none'

RUN_CONFIG = Config()
seed_everything(0)
print(asdict(RUN_CONFIG))

{'dataset': 'CIFAR10', 'n_tasks': 5, 'buffer_size': 200, 'epochs': 50, 'batch_size': 32, 'minibatch_size': 32, 'lr': 0.1, 'momentum': 0.9, 'weight_decay': 0.0005, 'empty_probability': 0.9, 'alpha': 0.5, 'beta': 0.5, 'class_balance': False, 'num_workers': 2, 'data_root': './data', 'gcil_mode': 'none'}


## 1. Class-incremental data stream

CIFAR-10 uses 5 tasks with 2 classes per task; CIFAR-100 uses 10 tasks with 10 classes per task. The task identity is not given to the model at test time.

In [32]:
def make_transforms(train=True, dataset='CIFAR10'):
    ops = [transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip()] if train else []
    mean,std=((0.5071,0.4867,0.4408),(0.2675,0.2565,0.2761)) if dataset.upper()=='CIFAR100' else ((0.4914,0.4822,0.4465),(0.2470,0.2435,0.2615))
    ops += [transforms.ToTensor(), transforms.Normalize(mean,std)]
    return transforms.Compose(ops)

class TinyImageNetVal(torch.utils.data.Dataset):
    def __init__(self, root, transform=None):
        from PIL import Image
        self.root=Path(root); self.transform=transform; self.records=[]
        wnids=(self.root/'wnids.txt').read_text().splitlines(); self.class_to_idx={c:i for i,c in enumerate(wnids)}
        ann=self.root/'val'/'val_annotations.txt'
        for line in ann.read_text().splitlines():
            fn, cls, *_ = line.split(); self.records.append((self.root/'val'/'images'/fn, self.class_to_idx[cls]))
    def __len__(self): return len(self.records)
    def __getitem__(self, i):
        from PIL import Image
        p,y=self.records[i]; x=Image.open(p).convert('RGB'); return (self.transform(x) if self.transform else x), y

class BenchmarkStream:
    """Paper datasets: Split CIFAR-10/100, Split TinyImageNet, and GCIL-CIFAR-100."""
    def __init__(self, cfg):
        name=cfg.dataset.upper(); self.cfg=cfg
        if name in ('CIFAR10','CIFAR100'):
            Dataset, self.classes = (datasets.CIFAR10,10) if name=='CIFAR10' else (datasets.CIFAR100,100)
            self.train=Dataset(cfg.data_root, train=True, download=True, transform=make_transforms(True,name)); self.test=Dataset(cfg.data_root, train=False, download=True, transform=make_transforms(False,name))
            self.train_targets=np.asarray(self.train.targets); self.test_targets=np.asarray(self.test.targets)
        elif name in ('TINYIMAGENET','TINY-IMAGENET'):
            root=Path(cfg.data_root)/'tiny-imagenet-200'; self.classes=200
            self.train=datasets.ImageFolder(root/'train', transform=transforms.Compose([transforms.RandomCrop(64,padding=4),transforms.RandomHorizontalFlip(),transforms.ToTensor(),transforms.Normalize((.4802,.4480,.3975),(.2770,.2691,.2821))]))
            self.test=TinyImageNetVal(root, transforms.Compose([transforms.ToTensor(),transforms.Normalize((.4802,.4480,.3975),(.2770,.2691,.2821))]))
            self.train_targets=np.asarray(self.train.targets); self.test_targets=np.asarray([y for _,y in self.test.records])
        else: raise ValueError('dataset must be CIFAR10, CIFAR100, or TinyImageNet')
        self.n_tasks=cfg.n_tasks; self.classes_per_task=self.classes//cfg.n_tasks; self.replay_transform=lambda x:x
        if name=='CIFAR100' and cfg.gcil_mode in ('uniform','longtail'):
            self.task_classes=self._make_gcil_classes()
        else: self.task_classes=[list(range(t*self.classes_per_task,(t+1)*self.classes_per_task)) for t in range(cfg.n_tasks)]

    def _make_gcil_classes(self):
        rng=np.random.default_rng(0); out=[]
        for t in range(self.n_tasks):
            # GCIL permits overlap; each task still observes a variable class subset.
            k=int(rng.integers(max(2,self.classes_per_task//2), self.classes_per_task+1)); out.append(sorted(rng.choice(self.classes,k,replace=False).tolist()))
        return out

    def loaders(self, task, batch_size, workers=2):
        cls=self.task_classes[task]; tr_idx=np.flatnonzero(np.isin(self.train_targets,cls)); te_idx=np.flatnonzero(np.isin(self.test_targets,cls))
        if self.cfg.dataset.upper()=='CIFAR100' and self.cfg.gcil_mode=='longtail':
            # Official GCIL long-tail weights use w_c = 0.984^c (the repository sampler).
            y=self.train_targets[tr_idx]; keep=[]; counts={c:max(10,int(5000*(0.984**c))) for c in cls}
            for c in cls: keep.extend(tr_idx[y==c][:counts[c]])
            tr_idx=np.asarray(keep)
        tr=DataLoader(Subset(self.train,tr_idx),batch_size=batch_size,shuffle=True,num_workers=workers,pin_memory=True)
        te=DataLoader(Subset(self.test,te_idx),batch_size=batch_size,shuffle=False,num_workers=workers,pin_memory=True)
        return tr,te

# Real paper data stream examples:
# stream = BenchmarkStream(Config(dataset='CIFAR10', n_tasks=5, epochs=50, buffer_size=200))
# stream = BenchmarkStream(Config(dataset='CIFAR100', n_tasks=10, epochs=50, buffer_size=500))
# stream = BenchmarkStream(Config(dataset='TinyImageNet', n_tasks=10, epochs=100, buffer_size=4000))
# stream = BenchmarkStream(Config(dataset='CIFAR100', n_tasks=10, gcil_mode='longtail', buffer_size=500))

In [33]:
class ReplayBuffer:
    """Reservoir replay with an optional class-balancing replacement rule."""
    def __init__(self, capacity, device, class_balance=False):
        self.capacity, self.device, self.class_balance = capacity, device, class_balance
        self.examples, self.labels = [], []
        self.seen = 0

    def __len__(self): return len(self.labels)
    def is_empty(self): return len(self) == 0

    def _index(self, label):
        if self.seen < self.capacity: return self.seen
        if self.class_balance and self.labels:
            counts = np.bincount(np.asarray(self.labels), minlength=int(max(self.labels))+1)
            candidates = np.flatnonzero(np.isin(self.labels, np.flatnonzero(counts == counts.max())))
            if np.random.randint(self.seen + 1) < self.capacity: return int(np.random.choice(candidates))
            return -1
        j = np.random.randint(self.seen + 1)
        return int(j) if j < self.capacity else -1

    def add(self, examples, labels):
        for x, y in zip(examples.detach().cpu(), labels.detach().cpu()):
            idx = self._index(int(y)); self.seen += 1
            if idx < 0: continue
            if idx == len(self.examples): self.examples.append(x.clone()); self.labels.append(int(y))
            else: self.examples[idx] = x.clone(); self.labels[idx] = int(y)

    def sample(self, n, transform):
        n = min(n, len(self)); ids = np.random.choice(len(self), n, replace=False)
        x = torch.stack([transform(self.examples[i]) for i in ids]).to(self.device)
        y = torch.tensor([self.labels[i] for i in ids], device=self.device)
        return x, y

In [34]:
class BasicBlock(nn.Module):
    expansion=1
    def __init__(self, cin, cout, stride=1):
        super().__init__(); self.conv1=nn.Conv2d(cin,cout,3,stride,1,bias=False); self.bn1=nn.BatchNorm2d(cout); self.conv2=nn.Conv2d(cout,cout,3,1,1,bias=False); self.bn2=nn.BatchNorm2d(cout)
        self.shortcut=nn.Identity() if stride==1 and cin==cout else nn.Sequential(nn.Conv2d(cin,cout,1,stride,bias=False),nn.BatchNorm2d(cout))
    def forward(self,x):
        out=F.relu(self.bn1(self.conv1(x))); out=self.bn2(self.conv2(out)); return F.relu(out+self.shortcut(x))

class IdempotentResNet(nn.Module):
    # Exact ResNet18_id2 split used by the public IDER implementation.
    def __init__(self, num_classes, nf=64):
        super().__init__(); self.num_classes=num_classes; self.feature_dim=nf*8
        self.stem=nn.Sequential(nn.Conv2d(3,nf,3,1,1,bias=False),nn.BatchNorm2d(nf),nn.ReLU())
        self.layer1=nn.Sequential(BasicBlock(nf,nf),BasicBlock(nf,nf)); self.layer2=nn.Sequential(BasicBlock(nf,nf*2,2),BasicBlock(nf*2,nf*2))
        # f1 stops at 128 channels and 16x16, exactly before f2's layer3.
        self.f1=nn.Sequential(self.stem,self.layer1,self.layer2)
        self.label_feature=nn.Sequential(nn.Linear(num_classes,128),nn.LeakyReLU())
        self.layer3=nn.Sequential(BasicBlock(nf*2,nf*4,2),BasicBlock(nf*4,nf*4)); self.layer4=nn.Sequential(BasicBlock(nf*4,nf*8,2),BasicBlock(nf*8,nf*8))
        self.classifier=nn.Linear(nf*8,num_classes)
    def f2_forward(self,z,second_input,return_features=False):
        z=z+self.label_feature(second_input)[...,None,None]; feat=self.layer4(self.layer3(z)); pooled=F.adaptive_avg_pool2d(feat,1).flatten(1)
        return (self.classifier(pooled),feat) if return_features else self.classifier(pooled)
    def forward(self,x,second_input,return_features=False): return self.f2_forward(self.f1(x),second_input,return_features)
    def image_features(self,x,second_input): return self.forward(x,second_input,return_features=True)[1]

    def first_pass(self, x):
        empty = torch.full((x.size(0), self.num_classes), 1/self.num_classes, device=x.device)
        return self(x, empty)

def one_hot_or_empty(labels, num_classes, p_empty, device):
    # The paper samples the empty/one-hot signal independently for every example.
    empty_mask = torch.rand(labels.size(0), device=device) < p_empty
    one_hot = F.one_hot(labels, num_classes=num_classes).float()
    empty = torch.full_like(one_hot, 1/num_classes)
    return torch.where(empty_mask[:, None], empty, one_hot)

In [35]:
def er_step(model, optimizer, x, y, buffer, cfg, transform):
    # Experience Replay baseline: one ordinary classifier pass per example.
    model.train(); optimizer.zero_grad(); C=model.num_classes; new_x,new_y=x,y; empty=torch.full((len(x),C),1/C,device=x.device)
    if not buffer.is_empty():
        bx,by=buffer.sample(cfg.minibatch_size,transform); x=torch.cat([x,bx]); y=torch.cat([y,by]); empty=torch.full((len(x),C),1/C,device=x.device)
    loss=F.cross_entropy(model(x,empty),y); loss.backward(); optimizer.step(); buffer.add(new_x,new_y)
    return {'total':float(loss.detach()),'L_ice':0.0,'L_ide':0.0,'L_rep-ice':float(loss.detach())}

def ider_step(model, old_model, optimizer, x, y, buffer, cfg, transform):
    model.train(); optimizer.zero_grad(); C = model.num_classes
    y_star = one_hot_or_empty(y, C, cfg.empty_probability, x.device)
    y0 = model(x, y_star); y1 = model(x, y0.softmax(-1))
    loss_ice = F.cross_entropy(y0, y) + F.cross_entropy(y1, y)
    loss_ide = torch.zeros((), device=x.device); loss_rep = torch.zeros((), device=x.device)

    if old_model is not None and not buffer.is_empty() and cfg.alpha:
        bx, by = buffer.sample(cfg.minibatch_size, transform)
        empty = torch.full((bx.size(0), C), 1/C, device=x.device)
        current_pred = model(bx, empty)
        with torch.no_grad(): stable_pred = old_model(bx, current_pred.softmax(-1))
        loss_ide = F.mse_loss(current_pred, stable_pred)

    if not buffer.is_empty() and cfg.beta:
        bx, by = buffer.sample(cfg.minibatch_size, transform)
        by_star = one_hot_or_empty(by, C, cfg.empty_probability, x.device)
        br0 = model(bx, by_star); br1 = model(bx, br0.softmax(-1))
        loss_rep = F.cross_entropy(br0, by) + F.cross_entropy(br1, by)

    total = loss_ice + cfg.alpha*loss_ide + cfg.beta*loss_rep
    total.backward(); optimizer.step(); buffer.add(x, y)
    return {
        'total': float(total.detach()), 'L_ice': float(loss_ice.detach()),
        'L_ide': float(loss_ide.detach()), 'L_rep-ice': float(loss_rep.detach())
    }

In [36]:
@torch.no_grad()
def evaluate(model, loaders, device):
    model.eval(); scores = []
    for loader in loaders:
        correct = total = 0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            pred = model.first_pass(x).argmax(1)
            correct += int((pred == y).sum()); total += y.numel()
        scores.append(100*correct/max(total,1))
    return scores

def expected_calibration_error(model, loader, device, bins=15):
    model.eval(); confs=[]; correct=[]
    with torch.no_grad():
        for x,y in loader:
            p = model.first_pass(x.to(device)).softmax(1); c, pred = p.max(1)
            confs.append(c.cpu()); correct.append(pred.cpu().eq(y))
    conf, cor = torch.cat(confs), torch.cat(correct).float(); ece = torch.zeros(())
    for lo, hi in zip(torch.linspace(0,1,bins+1)[:-1], torch.linspace(0,1,bins+1)[1:]):
        mask = (conf > lo) & (conf <= hi)
        if mask.any(): ece += mask.float().mean() * (cor[mask].mean() - conf[mask].mean()).abs()
    return float(ece*100)

def expected_calibration_error_seen(model, loaders, device, bins=15):
    # Calibration is measured on all test loaders seen so far, not only the last task.
    return float(np.mean([expected_calibration_error(model, loader, device, bins) for loader in loaders]))

def final_metrics(history):
    final = np.asarray(history[-1], dtype=float); faa = final.mean()
    # Histories are ragged: task j only exists from the moment it is learned.
    peaks = []
    for j, final_score in enumerate(final):
        observed = [row[j] for row in history if len(row) > j]
        peaks.append(max(observed))
    forgetting = np.mean(np.asarray(peaks) - final)
    return {'FAA': float(faa), 'FF': float(forgetting)}

In [37]:
def run_ider(cfg, stream=None, device=DEVICE):
    seed_everything(0)
    if stream is None: stream = BenchmarkStream(cfg)
    model = IdempotentResNet(stream.classes).to(device)
    old_model = None; buffer = ReplayBuffer(cfg.buffer_size, device, cfg.class_balance)
    history, loss_rows, test_loaders = [], [], []
    for task in range(cfg.n_tasks):
        train_loader, test_loader = stream.loaders(task, cfg.batch_size, cfg.num_workers)
        test_loaders.append(test_loader)
        optimizer = torch.optim.SGD(model.parameters(), lr=cfg.lr, momentum=cfg.momentum, weight_decay=cfg.weight_decay)
        name=cfg.dataset.upper(); scheduler=None if name=='CIFAR10' else torch.optim.lr_scheduler.MultiStepLR(optimizer,milestones=([35,45] if name=='CIFAR100' else [35,60,75]),gamma=0.1)
        for epoch in range(cfg.epochs):
            for x,y in tqdm(train_loader, desc=f'task {task+1}/{cfg.n_tasks}, epoch {epoch+1}', leave=False):
                x,y = x.to(device), y.to(device)
                row = ider_step(model, old_model, optimizer, x, y, buffer, cfg, stream.replay_transform)
                row.update(task=task, epoch=epoch); loss_rows.append(row)
            if scheduler is not None: scheduler.step()
        scores = evaluate(model, test_loaders, device); history.append(scores)
        print(f'task {task+1}: FAA over seen tasks = {np.mean(scores):.2f}% | scores = {[round(s,2) for s in scores]}')
        old_model = copy.deepcopy(model).eval()
        for p in old_model.parameters(): p.requires_grad_(False)
    metrics = final_metrics(history)
    metrics['ECE_seen'] = expected_calibration_error_seen(model, test_loaders, device)
    return model, history, pd.DataFrame(loss_rows), metrics

## 5. Paper training runs

The main path uses the real benchmark datasets below. No synthetic data is used for reported results.

In [38]:
# Split CIFAR-10: 5 tasks, 50 epochs/task, buffer 200 or 500.
cifar10_cfg = Config(dataset='CIFAR10', n_tasks=5, epochs=50, buffer_size=200)
# cifar10_stream = BenchmarkStream(cifar10_cfg)
# cifar10_model, cifar10_history, cifar10_losses, cifar10_metrics = run_ider(cifar10_cfg, cifar10_stream, DEVICE)

# Split CIFAR-100: 10 tasks, 50 epochs/task, buffer 500 or 2000.
cifar100_cfg = Config(dataset='CIFAR100', n_tasks=10, epochs=50, buffer_size=500)
# cifar100_stream = BenchmarkStream(cifar100_cfg)
# cifar100_model, cifar100_history, cifar100_losses, cifar100_metrics = run_ider(cifar100_cfg, cifar100_stream, DEVICE)

# Split TinyImageNet: 10 tasks, 100 epochs/task, buffer 4000.
tiny_cfg = Config(dataset='TinyImageNet', n_tasks=10, epochs=100, buffer_size=4000, batch_size=32)
# tiny_stream = BenchmarkStream(tiny_cfg)
# tiny_model, tiny_history, tiny_losses, tiny_metrics = run_ider(tiny_cfg, tiny_stream, DEVICE)

# GCIL-CIFAR-100: overlapping classes; switch gcil_mode to 'longtail' for imbalanced tasks.
gcil_cfg = Config(dataset='CIFAR100', n_tasks=10, epochs=50, buffer_size=500, gcil_mode='uniform')
# gcil_stream = BenchmarkStream(gcil_cfg)
# gcil_model, gcil_history, gcil_losses, gcil_metrics = run_ider(gcil_cfg, gcil_stream, DEVICE)

In [39]:
# Recommended starting run: one seed, one epoch, CIFAR-10.
full_cfg = Config(dataset='CIFAR10', n_tasks=5, buffer_size=200, epochs=50, class_balance=False)
# stream = BenchmarkStream(full_cfg)
# model, history, losses, metrics = run_ider(full_cfg, stream, DEVICE)
# print(metrics)
# pd.DataFrame(history, columns=[f'task_{i+1}' for i in range(len(history[-1]))]).plot(marker='o', ylim=(0,100), figsize=(8,4))
# plt.ylabel('Accuracy (%)'); plt.xlabel('Training task'); plt.title('IDER task accuracy'); plt.show()

## 6. BFP+ID and CLS-ER+ID (paper plug-ins)

Appendix D.3 defines `L_BFP = ||A h_t(x,0) - h_{t-1}(x,0)||²` and `L_BFP+ID = L_ice + αL_ide + βL_rep-ice + γL_BFP`. CLS-ER+ID keeps fast and slow EMA semantic memories and adds their confidence-selected consistency target.

In [40]:
class EMA:
    def __init__(self, model, decay): self.model=copy.deepcopy(model).eval(); self.decay=decay
    @torch.no_grad()
    def update(self, source):
        for q,p in zip(self.model.parameters(), source.parameters()): q.mul_(self.decay).add_(p, alpha=1-self.decay)
        for q,p in zip(self.model.buffers(), source.buffers()): q.copy_(p)

def bfp_id_step(model, old_model, A, optimizer, x, y, buffer, cfg, transform, gamma=1.0):
    model.train(); A.train(); optimizer.zero_grad(); C=model.num_classes
    ys=one_hot_or_empty(y,C,cfg.empty_probability,x.device); y0=model(x,ys); y1=model(x,y0.softmax(-1))
    l_ice=F.cross_entropy(y0,y)+F.cross_entropy(y1,y); l_ide=torch.zeros((),device=x.device); l_rep=torch.zeros((),device=x.device); l_bfp=torch.zeros((),device=x.device)
    if old_model is not None and not buffer.is_empty():
        bx,by=buffer.sample(cfg.minibatch_size,transform); empty=torch.full((len(bx),C),1/C,device=x.device); cur=model(bx,empty)
        with torch.no_grad(): stable=old_model(bx,cur.softmax(-1))
        h_cur=model.image_features(bx,empty).mean((2,3)); h_old=old_model.image_features(bx,empty).mean((2,3)).detach(); l_ide=F.mse_loss(cur,stable); l_bfp=torch.linalg.vector_norm(A(h_cur)-h_old,ord=2,dim=1).mean()
    if not buffer.is_empty():
        bx,by=buffer.sample(cfg.minibatch_size,transform); bs=one_hot_or_empty(by,C,cfg.empty_probability,x.device); r0=model(bx,bs); r1=model(bx,r0.softmax(-1)); l_rep=F.cross_entropy(r0,by)+F.cross_entropy(r1,by)
    total=l_ice+cfg.alpha*l_ide+cfg.beta*l_rep+gamma*l_bfp; total.backward(); optimizer.step(); buffer.add(x,y)
    return {'total':float(total.detach()),'L_ice':float(l_ice.detach()),'L_ide':float(l_ide.detach()),'L_rep-ice':float(l_rep.detach()),'L_BFP':float(l_bfp.detach())}

def clser_id_step(model, old_model, plastic, stable, optimizer, x, y, buffer, cfg, transform, lam=1.0):
    row=ider_step(model,old_model,optimizer,x,y,buffer,cfg,transform)
    if not buffer.is_empty():
        bx,by=buffer.sample(cfg.minibatch_size,transform); empty=torch.full((len(bx),model.num_classes),1/model.num_classes,device=x.device)
        with torch.no_grad(): pp=plastic.model(bx,empty); sp=stable.model(bx,empty); target=torch.where(pp.max(1,keepdim=True).values>=sp.max(1,keepdim=True).values,pp,sp)
        optimizer.zero_grad(); lc=F.mse_loss(model(bx,empty),target); (lam*lc).backward(); optimizer.step(); row['L_CLS-ER']=float(lc.detach())
    plastic.update(model); stable.update(model); return row

def run_variant(cfg, variant='ER+ID', stream=None, device=DEVICE, gamma=1.0):
    stream=BenchmarkStream(cfg) if stream is None else stream; model=IdempotentResNet(stream.classes).to(device); old=None; buffer=ReplayBuffer(cfg.buffer_size,device,cfg.class_balance); A=nn.Linear(model.feature_dim,model.feature_dim,bias=False).to(device) if variant=='BFP+ID' else None; plastic=stable=None; history=[]
    for task in range(cfg.n_tasks):
        tr,te=stream.loaders(task,cfg.batch_size,cfg.num_workers); opt_params=list(model.parameters()) + ([] if A is None else list(A.parameters())); lr=0.03 if variant=='BFP+ID' else cfg.lr; optimizer=torch.optim.SGD(opt_params,lr=lr,momentum=cfg.momentum,weight_decay=cfg.weight_decay); name=cfg.dataset.upper(); scheduler=None if name=='CIFAR10' else torch.optim.lr_scheduler.MultiStepLR(optimizer,milestones=[35,45] if name=='CIFAR100' else [35,60,75],gamma=.1)
        if variant=='CLS-ER+ID' and plastic is None: plastic=EMA(model,.999); stable=EMA(model,.9999)
        for epoch in range(cfg.epochs):
            for x,y in tqdm(tr,desc=f'{variant} task {task+1}/{cfg.n_tasks}',leave=False):
                x,y=x.to(device),y.to(device)
                if variant=='ER': row=er_step(model,optimizer,x,y,buffer,cfg,stream.replay_transform)
                elif variant=='BFP+ID': row=bfp_id_step(model,old,A,optimizer,x,y,buffer,cfg,stream.replay_transform,gamma)
                elif variant=='CLS-ER+ID': row=clser_id_step(model,old,plastic,stable,optimizer,x,y,buffer,cfg,stream.replay_transform)
                else: row=ider_step(model,old,optimizer,x,y,buffer,cfg,stream.replay_transform)
            if scheduler is not None: scheduler.step()
        test_loaders=[stream.loaders(i,cfg.batch_size,cfg.num_workers)[1] for i in range(task+1)]; scores=evaluate(model,test_loaders,device); history.append(scores); print(variant,'task',task+1,'FAA',round(float(np.mean(scores)),2)); old=copy.deepcopy(model).eval()
        for p in old.parameters(): p.requires_grad_(False)
    metrics=final_metrics(history); metrics['ECE_seen']=expected_calibration_error_seen(model,test_loaders,device); return model,history,metrics

In [41]:
def run_ablation(base_cfg, stream=None, device=DEVICE):
    rows=[]
    for name in ['ER','ER+ID','BFP+ID','CLS-ER+ID']:
        _,_,metrics=run_variant(copy.deepcopy(base_cfg),variant=name,stream=stream,device=device)
        rows.append({'method':name,**metrics})
    return pd.DataFrame(rows)

# Real-data benchmark examples; uncomment only after setting the matching dataset path.
# result_bfp = run_variant(cifar100_cfg,'BFP+ID',cifar100_stream,DEVICE,gamma=1.0)
# result_clser = run_variant(cifar100_cfg,'CLS-ER+ID',cifar100_stream,DEVICE)
# display(run_ablation(cifar100_cfg,cifar100_stream,DEVICE))

In [42]:
def run_five_seeds(cfg, variant='ER+ID', device=DEVICE, gamma=1.0):
    """Paper protocol: five independent seeds, then mean +/- std."""
    rows=[]
    for seed in range(5):
        seed_everything(seed); stream=BenchmarkStream(copy.deepcopy(cfg))
        _,history,metrics=run_variant(copy.deepcopy(cfg),variant=variant,stream=stream,device=device,gamma=gamma)
        rows.append({'seed':seed,'variant':variant,**metrics})
    out=pd.DataFrame(rows); summary=out[['FAA','FF']].agg(['mean','std']).T
    return out,summary

# Table-1-style calls (long GPU runs; use the exact paper buffer per dataset):
# cifar10_runs, cifar10_summary=run_five_seeds(Config(dataset='CIFAR10',n_tasks=5,epochs=50,buffer_size=200),'ER+ID')
# cifar100_runs, cifar100_summary=run_five_seeds(Config(dataset='CIFAR100',n_tasks=10,epochs=50,buffer_size=500),'ER+ID')
# tiny_runs, tiny_summary=run_five_seeds(Config(dataset='TinyImageNet',n_tasks=10,epochs=100,buffer_size=4000),'ER+ID')
# bfp_runs, bfp_summary=run_five_seeds(Config(dataset='CIFAR100',n_tasks=10,epochs=50,buffer_size=500,class_balance=True),'BFP+ID',gamma=1.0)

In [43]:
# ===================== RUN CONTROL =====================
# Keep False while checking the notebook. Set True when the kernel is connected and you are ready.
RUN_TRAINING = False
RUN_VARIANT = 'ER+ID'       # ER, ER+ID, BFP+ID, CLS-ER+ID
RUN_DATASET = 'CIFAR10'     # CIFAR10, CIFAR100, TinyImageNet
RUN_BUFFER = 200
RUN_TASKS = 5
RUN_EPOCHS = 50
RUN_FIVE_SEEDS = False      # True reproduces the paper's 5-seed protocol

if RUN_TRAINING:
    run_cfg = Config(dataset=RUN_DATASET,n_tasks=RUN_TASKS,epochs=RUN_EPOCHS,buffer_size=RUN_BUFFER,class_balance=(RUN_VARIANT=='BFP+ID'))
    if RUN_FIVE_SEEDS:
        seed_runs, seed_summary = run_five_seeds(run_cfg, RUN_VARIANT, DEVICE, gamma=1.0)
        display(seed_runs, seed_summary)
    else:
        run_stream = BenchmarkStream(run_cfg)
        _, run_history, run_metrics = run_variant(run_cfg, RUN_VARIANT, run_stream, DEVICE, gamma=1.0)
        print(run_metrics)
        display(pd.DataFrame(run_history))

In [ ]:
# ---------------- PAPER TRAINING: RUN ALL CELLS ----------------
# Change only these settings before pressing Run All.
TRAIN_VARIANT = 'ER+ID'       # ER, ER+ID, BFP+ID, CLS-ER+ID
TRAIN_DATASET = 'CIFAR10'     # CIFAR10, CIFAR100, TinyImageNet
TRAIN_TASKS = 5
TRAIN_EPOCHS = 50
TRAIN_BUFFER = 200
TRAIN_SEED = 0

# Cell step 1 — real benchmark data
seed_everything(TRAIN_SEED)
train_cfg = Config(dataset=TRAIN_DATASET,n_tasks=TRAIN_TASKS,epochs=TRAIN_EPOCHS,buffer_size=TRAIN_BUFFER,class_balance=(TRAIN_VARIANT=='BFP+ID'))
train_stream = BenchmarkStream(train_cfg)
print('dataset:',TRAIN_DATASET,'tasks:',TRAIN_TASKS,'epochs/task:',TRAIN_EPOCHS,'buffer:',TRAIN_BUFFER,'device:',DEVICE)

# Cell step 2 — exact IDER model, frozen previous model, replay buffer
trained_model = IdempotentResNet(train_stream.classes).to(DEVICE)
previous_model = None
replay = ReplayBuffer(TRAIN_BUFFER,DEVICE,train_cfg.class_balance)
projection_A = nn.Linear(trained_model.feature_dim,trained_model.feature_dim,bias=False).to(DEVICE) if TRAIN_VARIANT=='BFP+ID' else None
plastic_memory = stable_memory = None
train_history=[]; train_loss_rows=[]; seen_test_loaders=[]

# Cell step 3 — task-by-task continual learning
for task_id in range(TRAIN_TASKS):
    train_loader,test_loader=train_stream.loaders(task_id,train_cfg.batch_size,train_cfg.num_workers)
    seen_test_loaders.append(test_loader)
    params=list(trained_model.parameters()) + ([] if projection_A is None else list(projection_A.parameters()))
    lr=0.03 if TRAIN_VARIANT=='BFP+ID' else train_cfg.lr
    optimizer=torch.optim.SGD(params,lr=lr,momentum=train_cfg.momentum,weight_decay=train_cfg.weight_decay)
    dataset_name=TRAIN_DATASET.upper()
    milestones=None if dataset_name=='CIFAR10' else ([35,45] if dataset_name=='CIFAR100' else [35,60,75])
    scheduler=None if milestones is None else torch.optim.lr_scheduler.MultiStepLR(optimizer,milestones=milestones,gamma=0.1)
    if TRAIN_VARIANT=='CLS-ER+ID' and plastic_memory is None:
        plastic_memory=EMA(trained_model,0.999); stable_memory=EMA(trained_model,0.9999)
    for epoch_id in range(TRAIN_EPOCHS):
        trained_model.train()
        progress=tqdm(train_loader,desc=f'{TRAIN_VARIANT} | task {task_id+1}/{TRAIN_TASKS} | epoch {epoch_id+1}/{TRAIN_EPOCHS}',leave=False)
        for images,labels in progress:
            images,labels=images.to(DEVICE),labels.to(DEVICE)
            if TRAIN_VARIANT=='ER':
                loss_row=er_step(trained_model,optimizer,images,labels,replay,train_cfg,train_stream.replay_transform)
            elif TRAIN_VARIANT=='BFP+ID':
                loss_row=bfp_id_step(trained_model,previous_model,projection_A,optimizer,images,labels,replay,train_cfg,train_stream.replay_transform,gamma=1.0)
            elif TRAIN_VARIANT=='CLS-ER+ID':
                loss_row=clser_id_step(trained_model,previous_model,plastic_memory,stable_memory,optimizer,images,labels,replay,train_cfg,train_stream.replay_transform)
            else:
                loss_row=ider_step(trained_model,previous_model,optimizer,images,labels,replay,train_cfg,train_stream.replay_transform)
            loss_row.update(task=task_id,epoch=epoch_id); train_loss_rows.append(loss_row); progress.set_postfix(loss=loss_row['total'])
        if scheduler is not None: scheduler.step()
    # Cell step 4 — evaluate every task seen so far
    scores=evaluate(trained_model,seen_test_loaders,DEVICE); train_history.append(scores)
    print(f'task {task_id+1}: FAA={np.mean(scores):.2f}% | scores={[round(s,2) for s in scores]}')
    previous_model=copy.deepcopy(trained_model).to(DEVICE).eval()
    for p in previous_model.parameters(): p.requires_grad_(False)

# Cell step 5 — paper metrics and checkpoint
train_metrics=final_metrics(train_history)
train_metrics['ECE_seen']=expected_calibration_error_seen(trained_model,seen_test_loaders,DEVICE)
loss_table=pd.DataFrame(train_loss_rows)
print('FINAL METRICS:',train_metrics)
display(pd.DataFrame(train_history)); display(loss_table.tail())
checkpoint_path=f'{TRAIN_VARIANT}_{TRAIN_DATASET}_seed{TRAIN_SEED}.pt'
torch.save({'model':trained_model.state_dict(),'config':asdict(train_cfg),'history':train_history,'metrics':train_metrics},checkpoint_path)
print('checkpoint saved:',checkpoint_path)

In [ ]:
# Step 1 — exact paper experiment matrix
PAPER_EXPERIMENTS = {
    'CIFAR10_b200': Config(dataset='CIFAR10',n_tasks=5,epochs=50,buffer_size=200,lr=0.1),
    'CIFAR10_b500': Config(dataset='CIFAR10',n_tasks=5,epochs=50,buffer_size=500,lr=0.1),
    'CIFAR100_b500': Config(dataset='CIFAR100',n_tasks=10,epochs=50,buffer_size=500,lr=0.1),
    'CIFAR100_b2000': Config(dataset='CIFAR100',n_tasks=10,epochs=50,buffer_size=2000,lr=0.1),
    'TinyImageNet_b4000': Config(dataset='TinyImageNet',n_tasks=10,epochs=100,buffer_size=4000,lr=0.1),
}
RESULT_DIR=Path('paper_results'); RESULT_DIR.mkdir(exist_ok=True)
print(pd.DataFrame([{'name':k,**asdict(v)} for k,v in PAPER_EXPERIMENTS.items()]))

In [ ]:
# Step 2 — one reproducible real-data experiment
def run_paper_experiment(name, variant='ER+ID', seed=0, alpha=None, beta=None):
    cfg=copy.deepcopy(PAPER_EXPERIMENTS[name]); cfg.alpha=cfg.alpha if alpha is None else alpha; cfg.beta=cfg.beta if beta is None else beta; cfg.class_balance=(variant=='BFP+ID')
    seed_everything(seed); stream=BenchmarkStream(cfg)
    _,history,metrics=run_variant(cfg,variant=variant,stream=stream,device=DEVICE,gamma=1.0)
    row={'experiment':name,'variant':variant,'seed':seed,'alpha':cfg.alpha,'beta':cfg.beta,**metrics}
    pd.DataFrame([row]).to_csv(RESULT_DIR/f'{name}_{variant}_seed{seed}.csv',index=False)
    return row,history

In [ ]:
SEEDS=range(5)  # paper protocol: five independent runs
cifar10_results=[]
for exp in ['CIFAR10_b200','CIFAR10_b500']:
    for method in ['ER','ER+ID']:
        for seed in SEEDS:
            row,_=run_paper_experiment(exp,method,seed); cifar10_results.append(row); print(row)
cifar10_table=pd.DataFrame(cifar10_results); display(cifar10_table)
cifar10_table.to_csv(RESULT_DIR/'cifar10_er_vs_id.csv',index=False)

## ER vs ER+ID — Split CIFAR-100

This cell runs the paper's 10-task CIFAR-100 protocol for buffer 500 and 2000.

In [ ]:
cifar100_results=[]
for exp in ['CIFAR100_b500','CIFAR100_b2000']:
    for method in ['ER','ER+ID']:
        for seed in SEEDS:
            row,_=run_paper_experiment(exp,method,seed); cifar100_results.append(row); print(row)
cifar100_table=pd.DataFrame(cifar100_results); display(cifar100_table)
cifar100_table.to_csv(RESULT_DIR/'cifar100_er_vs_id.csv',index=False)

In [ ]:
tiny_results=[]
for method in ['ER','ER+ID']:
    for seed in SEEDS:
        row,_=run_paper_experiment('TinyImageNet_b4000',method,seed); tiny_results.append(row); print(row)
tiny_table=pd.DataFrame(tiny_results); display(tiny_table)
tiny_table.to_csv(RESULT_DIR/'tinyimagenet_er_vs_id.csv',index=False)

## Component ablation — SIM / IDM / ER

This follows Table 8: SIM only, SIM+ER, and SIM+IDM+ER.

In [ ]:
ablation_results=[]
for label,alpha,beta in [('SIM-only',0.0,0.0),('SIM+ER',0.0,0.5),('SIM+IDM+ER',0.5,0.5)]:
    row,_=run_paper_experiment('CIFAR100_b500','ER+ID',0,alpha=alpha,beta=beta); row['ablation']=label; ablation_results.append(row); print(row)
ablation_table=pd.DataFrame(ablation_results); display(ablation_table); ablation_table.to_csv(RESULT_DIR/'component_ablation.csv',index=False)

In [ ]:
plugin_results=[]
for method in ['BFP+ID','CLS-ER+ID']:
    row,_=run_paper_experiment('CIFAR100_b500',method,0); plugin_results.append(row); print(row)
plugin_table=pd.DataFrame(plugin_results); display(plugin_table); plugin_table.to_csv(RESULT_DIR/'plugin_ablation.csv',index=False)